# Build MSOA Modelling Dataset

The aim of this notebook is to combine the processed MSOA-level spatial data, including population estimates, public transport nodes and the bakery provision. Until the final bakery dataset is produced we will be using provisional bakery counts and these are only to test the downstream data pipeline.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


SPATIAL_FOLDER = Path("../data/spatial/processed")
BUSINESS_FOLDER = Path("../data/business/interim")

MSOA_PATH = SPATIAL_FOLDER / "london_msoa_2021.gpkg"
POPULATION_PATH = SPATIAL_FOLDER / "london_msoa_population_2024.csv"
TRANSPORT_PATH = SPATIAL_FOLDER / "london_msoa_transport_nodes.csv"

FEATURES_OUTPUT_PATH = SPATIAL_FOLDER / "london_msoa_features.csv"

In [ ]:
msoa = gpd.read_file(MSOA_PATH, 
                     columns = ["msoa_code", "msoa_name", "borough_code", "borough_name", "area_km2"], 
                     ignore_geometry= True)

population = pd.read_csv(POPULATION_PATH,
                         usecols= ["msoa_code", "population"])

transport = pd.read_csv(TRANSPORT_PATH,
                        usecols= ["msoa_code", "transport_nodes"])

print(f"MSOA rows: {len(msoa)}")
print(f"Population rows: {len(population)}")
print(f"Transport rows: {len(transport)}")

# Combine MSOA features

msoa_features = (msoa.merge(population, on="msoa_code", how="left", validate="one_to_one"
    ).merge(transport, on="msoa_code", how="left", validate="one_to_one"
    ).sort_values("msoa_code")
    .reset_index(drop=True))

print(f"Combined MSOA rows: {len(msoa_features)}")

msoa_features.head()

## Validate combined MSOA features

print(f"MSOA rows: {len(msoa_features)}")
print(f"Unique MSOA codes: {msoa_features['msoa_code'].nunique()}")
print(f"Duplicate MSOA codes: {msoa_features['msoa_code'].duplicated().sum()}")

print(f"\nMissing area values: {msoa_features['area_km2'].isna().sum()}")
print(f"Missing population values: {msoa_features['population'].isna().sum()}")
print(f"Missing transport counts: {msoa_features['transport_nodes'].isna().sum()}")

print(f"\nNon-positive areas: {(msoa_features['area_km2'] <= 0).sum()}")
print(f"Non-positive populations: {(msoa_features['population'] <= 0).sum()}")
print(f"Negative transport counts: {(msoa_features['transport_nodes'] < 0).sum()}")